# Calendar Dimension

Builds `caresyncsydatabricks.silver.calendar` with one row per calendar day from 1940-01-01 to 2030-12-31, including common date attributes for joins and reporting.

In [0]:
from pyspark.sql.functions import *

START_DATE = "1940-01-01"
END_DATE = "2030-12-31"

calendar_df = (
    spark.sql(f"SELECT explode(sequence(DATE'{START_DATE}', DATE'{END_DATE}', INTERVAL 1 DAY)) AS calendar_date")
    .withColumn("date_key", date_format("calendar_date", "yyyyMMdd").cast("int"))
    .withColumn("year", year("calendar_date"))
    .withColumn("quarter", quarter("calendar_date"))
    .withColumn("quarter_name", concat(lit("Q"), col("quarter")))
    .withColumn("month", month("calendar_date"))
    .withColumn("month_name", date_format("calendar_date", "MMMM"))
    .withColumn("month_short_name", date_format("calendar_date", "MMM"))
    .withColumn("year_month", date_format("calendar_date", "yyyy-MM"))
    .withColumn("day_of_month", dayofmonth("calendar_date"))
    .withColumn("day_of_year", dayofyear("calendar_date"))
    .withColumn("day_of_week", dayofweek("calendar_date"))  # 1 = Sunday ... 7 = Saturday
    .withColumn("day_name", date_format("calendar_date", "EEEE"))
    .withColumn("day_short_name", date_format("calendar_date", "EEE"))
    .withColumn("week_of_year", weekofyear("calendar_date"))  # ISO week
    .withColumn("is_weekend", col("day_of_week").isin(1, 7))
    .withColumn("month_start_date", trunc("calendar_date", "month"))
    .withColumn("month_end_date", last_day("calendar_date"))
    .withColumn("quarter_start_date", date_trunc("quarter", "calendar_date").cast("date"))
    .withColumn("year_start_date", trunc("calendar_date", "year"))
    .withColumn("is_month_end", col("calendar_date") == col("month_end_date"))
    .withColumn("is_leap_year", ((col("year") % 4 == 0) & (col("year") % 100 != 0)) | (col("year") % 400 == 0))
    .select(
        "date_key", "calendar_date", "year", "quarter", "quarter_name", "month", "month_name",
        "month_short_name", "year_month", "day_of_month", "day_of_year", "day_of_week", "day_name",
        "day_short_name", "week_of_year", "is_weekend", "month_start_date", "month_end_date",
        "quarter_start_date", "year_start_date", "is_month_end", "is_leap_year"
    )
)

display(calendar_df.limit(10))

In [0]:
(
    calendar_df.write
    .mode("error")  # creates the table; fails safely if it already exists
    .saveAsTable("caresyncsydatabricks.silver.calendar")
)

In [0]:
cal = spark.read.table("caresyncsydatabricks.silver.calendar")
display(
    cal.agg(
        count("*").alias("row_count"),
        countDistinct("calendar_date").alias("distinct_dates"),
        min("calendar_date").alias("min_date"),
        max("calendar_date").alias("max_date"),
        (datediff(max("calendar_date"), min("calendar_date")) + 1).alias("expected_days")
    )
)